In [4]:
#load bounding_box_infos.json
import json
from itertools import islice

with open('bounding_box_infos.json', 'r') as f:
    bounding_box_infos = json.load(f)
print("Sample bounding box infos:")
for key, value in islice(bounding_box_infos.items(), 3):
    print(f"\t{key, value}")

Sample bounding box infos:
	('2013-02-22_06_05_00.jpg', [{'center_x': 317.8463134765625, 'center_y': 69.99588012695312, 'width': 79.760009765625, 'height': 44.18255615234375, 'confidence': 0.4845370054244995, 'label': 'car', 'frame': 0}, {'center_x': 969.0333862304688, 'center_y': 453.38128662109375, 'width': 59.8212890625, 'height': 41.798583984375, 'confidence': 0.2918429672718048, 'label': 'car', 'frame': 0}, {'center_x': 517.2152709960938, 'center_y': 347.97894287109375, 'width': 164.34130859375, 'height': 88.1417236328125, 'confidence': 0.25580349564552307, 'label': 'car', 'frame': 0}, {'center_x': 982.3997192382812, 'center_y': 459.46673583984375, 'width': 31.6905517578125, 'height': 23.9658203125, 'confidence': 0.2511134147644043, 'label': 'car', 'frame': 0}])
	('2013-02-22_06_10_00.jpg', [{'center_x': 318.3199462890625, 'center_y': 68.77516174316406, 'width': 77.49981689453125, 'height': 42.964813232421875, 'confidence': 0.5844017863273621, 'label': 'car', 'frame': 1}, {'center

In [ ]:
#load clusters from all_clusters.json
with open('all_clusters.json', 'r') as f:
    all_clusters = json.load(f)
#print sample clusters
print("Sample clusters:")
for key, value in islice(all_clusters.items(), 3):
    print(f"\t{key}: {value}")

Sample clusters:
	2013-02-22: {'0': {'center': ['1194.48', '311.99'], 'radius': '2.98'}, '1': {'center': ['355.05', '530.47'], 'radius': '2.26'}, '2': {'center': ['733.22', '205.35'], 'radius': '1.97'}, '3': {'center': ['789.15', '521.68'], 'radius': '15.54'}, '4': {'center': ['1107.75', '252.66'], 'radius': '7.94'}, '5': {'center': ['1112.28', '155.11'], 'radius': '2.49'}, '6': {'center': ['16.63', '95.71'], 'radius': '2.45'}, '7': {'center': ['610.89', '297.84'], 'radius': '6.46'}, '8': {'center': ['510.19', '195.70'], 'radius': '3.64'}, '9': {'center': ['1208.76', '214.14'], 'radius': '1.98'}, '10': {'center': ['1129.24', '366.98'], 'radius': '6.59'}, '11': {'center': ['809.82', '273.32'], 'radius': '16.83'}, '12': {'center': ['1033.15', '204.85'], 'radius': '11.24'}, '13': {'center': ['887.85', '330.66'], 'radius': '3.36'}, '14': {'center': ['588.03', '480.91'], 'radius': '3.53'}, '15': {'center': ['283.17', '359.99'], 'radius': '1.30'}, '16': {'center': ['731.15', '432.08'], 'radi

In [ ]:
import pandas as pd
#iterate through dates, which are keys in all_clusters
training_data_centers = {}
training_data_bboxes = {}

training_df = pd.DataFrame()
training_rows = []

for date, clusters in all_clusters.items():
    print(f"Processing date: {date} with {len(clusters)} clusters")
    
    #get all images for this date
    date_images = [img for img in bounding_box_infos.keys() if img.startswith(date)]
    training_data_centers[date] = {}
    training_data_bboxes[date] = {}
    
    print(f"\tFound {len(date_images)} images for this date.")
    for cluster_number in clusters.keys():
        cluster_data = clusters[cluster_number]
        cluster_center = cluster_data['center']
        cluster_x = round(float(cluster_center[0]))
        cluster_y = round(float(cluster_center[1]))
        cluster_key_repr = f"({cluster_x},{cluster_y})"
        
        cluster_width = 0 #to fill in
        cluster_height = 0 #to fill in
        
        for img in date_images:
            # Ensure dictionaries are only initialized once per image
            if img not in training_data_centers[date]:
                training_data_centers[date][img] = {}
            if img not in training_data_bboxes[date]:
                training_data_bboxes[date][img] = {}
            
            bbox_info = bounding_box_infos[img]
            found = False
            for bbox in bbox_info:
                #print(f"\t\t\tChecking bbox: {bbox}")
                bbox_center_x = round(float(bbox['center_x']))
                bbox_center_y = round(float(bbox['center_y']))
                bbox_width = round(float(bbox['width']))
                bbox_height = round(float(bbox['height']))
                
                #Check if bbox center is within cluster center +/- half width/height
                if (cluster_x - bbox_width / 2 <= bbox_center_x <= cluster_x + bbox_width / 2 and
                    cluster_y - bbox_height / 2 <= bbox_center_y <= cluster_y + bbox_height / 2):
                    #print(f"\t\tMatch found in image {img}")
                    #this should have the label 1 in training data
                    training_data_centers[date][img][cluster_key_repr] = 1
                    #print(" - Match found, labeled 1")
                    found = True
                    cluster_width = bbox_width
                    cluster_height = bbox_height
                    break #no need to check other bboxes in this image
            if not found:
                training_data_centers[date][img][cluster_key_repr] = 0
                
        #See if cluster has ever been found
        if cluster_width == 0 or cluster_height == 0:
            print(f"\tWarning: Cluster {cluster_number} at {cluster_key_repr} was never matched in any image.")
        else:
            #We have cluster dimensions, reformat cluster key to be [x_min, y_min, x_max, y_max]
            x_min = int(cluster_x - cluster_width / 2)
            y_min = int(cluster_y - cluster_height / 2)
            x_max = int(cluster_x + cluster_width / 2)
            y_max = int(cluster_y + cluster_height / 2)
            new_cluster_key_repr = f"[{x_min},{y_min},{x_max},{y_max}]"
            #print(f"\tCluster {cluster_number} key reformatted from {cluster_key_repr} to {new_cluster_key_repr}")
            #Update training data keys
            for date in training_data_centers.keys():
                for img in training_data_centers[date].keys():
                    if cluster_key_repr in training_data_centers[date][img].keys():
                        training_data_bboxes[date][img][new_cluster_key_repr] = training_data_centers[date][img][cluster_key_repr]
            

Processing date: 2013-02-22 with 53 clusters
	Found 154 images for this date.
Processing date: 2013-04-08 with 4 clusters
	Found 4 images for this date.
Processing date: 2013-03-13 with 45 clusters
	Found 137 images for this date.
Processing date: 2013-03-19 with 32 clusters
	Found 33 images for this date.
Processing date: 2013-03-18 with 49 clusters
	Found 135 images for this date.
Processing date: 2013-03-03 with 0 clusters
	Found 33 images for this date.
Processing date: 2013-04-09 with 26 clusters
	Found 33 images for this date.
Processing date: 2013-03-22 with 19 clusters
	Found 29 images for this date.
Processing date: 2013-03-14 with 40 clusters
	Found 108 images for this date.
Processing date: 2013-03-10 with 0 clusters
	Found 125 images for this date.
Processing date: 2013-03-21 with 38 clusters
	Found 66 images for this date.
Processing date: 2013-04-13 with 6 clusters
	Found 58 images for this date.
Processing date: 2013-02-23 with 8 clusters
	Found 26 images for this date.


In [99]:
training_data_centers

{'2013-02-22': {'2013-02-22_06_05_00.jpg': {'(341,105)': 0},
  '2013-02-22_06_10_00.jpg': {'(341,105)': 0},
  '2013-02-22_06_15_00.jpg': {'(341,105)': 0},
  '2013-02-22_06_20_00.jpg': {'(341,105)': 0},
  '2013-02-22_06_25_00.jpg': {'(341,105)': 0},
  '2013-02-22_06_35_00.jpg': {'(341,105)': 0},
  '2013-02-22_06_40_00.jpg': {'(341,105)': 0},
  '2013-02-22_06_45_00.jpg': {'(341,105)': 0},
  '2013-02-22_06_50_00.jpg': {'(341,105)': 0},
  '2013-02-22_06_55_00.jpg': {'(341,105)': 0},
  '2013-02-22_07_00_01.jpg': {'(341,105)': 0},
  '2013-02-22_07_05_01.jpg': {'(341,105)': 0},
  '2013-02-22_07_10_01.jpg': {'(341,105)': 1},
  '2013-02-22_07_15_01.jpg': {'(341,105)': 1},
  '2013-02-22_07_20_01.jpg': {'(341,105)': 1},
  '2013-02-22_07_25_01.jpg': {'(341,105)': 1},
  '2013-02-22_07_30_01.jpg': {'(341,105)': 1},
  '2013-02-22_07_35_01.jpg': {'(341,105)': 1},
  '2013-02-22_07_40_01.jpg': {'(341,105)': 1},
  '2013-02-22_07_45_01.jpg': {'(341,105)': 1},
  '2013-02-22_07_50_01.jpg': {'(341,105)': 1},

In [100]:
training_data_bboxes

{'2013-02-22': {'2013-02-22_06_05_00.jpg': {'[304,84,377,126]': 0},
  '2013-02-22_06_10_00.jpg': {'[304,84,377,126]': 0},
  '2013-02-22_06_15_00.jpg': {'[304,84,377,126]': 0},
  '2013-02-22_06_20_00.jpg': {'[304,84,377,126]': 0},
  '2013-02-22_06_25_00.jpg': {'[304,84,377,126]': 0},
  '2013-02-22_06_35_00.jpg': {'[304,84,377,126]': 0},
  '2013-02-22_06_40_00.jpg': {'[304,84,377,126]': 0},
  '2013-02-22_06_45_00.jpg': {'[304,84,377,126]': 0},
  '2013-02-22_06_50_00.jpg': {'[304,84,377,126]': 0},
  '2013-02-22_06_55_00.jpg': {'[304,84,377,126]': 0},
  '2013-02-22_07_00_01.jpg': {'[304,84,377,126]': 0},
  '2013-02-22_07_05_01.jpg': {'[304,84,377,126]': 0},
  '2013-02-22_07_10_01.jpg': {'[304,84,377,126]': 1},
  '2013-02-22_07_15_01.jpg': {'[304,84,377,126]': 1},
  '2013-02-22_07_20_01.jpg': {'[304,84,377,126]': 1},
  '2013-02-22_07_25_01.jpg': {'[304,84,377,126]': 1},
  '2013-02-22_07_30_01.jpg': {'[304,84,377,126]': 1},
  '2013-02-22_07_35_01.jpg': {'[304,84,377,126]': 1},
  '2013-02-22_

In [102]:
# Rearrange training_data_bboxes to have image names as top-level keys
rearranged_bboxes = {}

for date, images in training_data_bboxes.items():
    for image_name, bboxes in images.items():
        if image_name not in rearranged_bboxes:
            rearranged_bboxes[image_name] = {}
        for bbox_key, label in bboxes.items():
            rearranged_bboxes[image_name][bbox_key] = label

In [110]:
list(rearranged_bboxes.keys())

['2013-02-22_06_05_00.jpg',
 '2013-02-22_06_10_00.jpg',
 '2013-02-22_06_15_00.jpg',
 '2013-02-22_06_20_00.jpg',
 '2013-02-22_06_25_00.jpg',
 '2013-02-22_06_35_00.jpg',
 '2013-02-22_06_40_00.jpg',
 '2013-02-22_06_45_00.jpg',
 '2013-02-22_06_50_00.jpg',
 '2013-02-22_06_55_00.jpg',
 '2013-02-22_07_00_01.jpg',
 '2013-02-22_07_05_01.jpg',
 '2013-02-22_07_10_01.jpg',
 '2013-02-22_07_15_01.jpg',
 '2013-02-22_07_20_01.jpg',
 '2013-02-22_07_25_01.jpg',
 '2013-02-22_07_30_01.jpg',
 '2013-02-22_07_35_01.jpg',
 '2013-02-22_07_40_01.jpg',
 '2013-02-22_07_45_01.jpg',
 '2013-02-22_07_50_01.jpg',
 '2013-02-22_07_55_02.jpg',
 '2013-02-22_08_00_02.jpg',
 '2013-02-22_08_05_02.jpg',
 '2013-02-22_08_10_02.jpg',
 '2013-02-22_08_15_02.jpg',
 '2013-02-22_08_20_02.jpg',
 '2013-02-22_08_25_02.jpg',
 '2013-02-22_08_30_02.jpg',
 '2013-02-22_08_35_02.jpg',
 '2013-02-22_08_40_02.jpg',
 '2013-02-22_08_45_02.jpg',
 '2013-02-22_08_50_02.jpg',
 '2013-02-22_08_55_02.jpg',
 '2013-02-22_09_00_03.jpg',
 '2013-02-22_09_05_0